In [ ]:
# Data handling and utilities
import pandas as pd
import numpy as np
import time
import psutil
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-learn tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support
)

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# TensorFlow / Keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping

# Energy measurement
import pyRAPL


In [ ]:
# === Step 1: Load and preprocess the data ===
from scipy.signal import savgol_filter

# SNV normalization function
def snv(input_data):
    """
    Standard Normal Variate (SNV) normalization
    """
    return (input_data - np.mean(input_data, axis=1, keepdims=True)) / np.std(input_data, axis=1, keepdims=True)

df = pd.read_csv('Dataset _Raman.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

# Savitzky–Golay first derivative (25 points)
X_sg = savgol_filter(X.values, window_length=25, polyorder=2, deriv=1, axis=1)

# SNV normalization
X_snv = snv(X_sg)

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snv)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# === Define 1D CNN Model (with best hyperparameters) ===
class Raman1DCNN(nn.Module):
    def __init__(self, input_length, num_classes):
        super(Raman1DCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten_dim = (input_length // 4) * 64
        self.fc1 = nn.Linear(self.flatten_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

        self.dropout = nn.Dropout(0.43870)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# === Train & Evaluate function ===
def train_evaluate(split_ratio):
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=split_ratio, random_state=42)

    # Convert to tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)

    # Model init
    input_length = X.shape[1]
    num_classes = len(np.unique(y_encoded))
    model = Raman1DCNN(input_length, num_classes)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.00057665)

    epochs = 30

    # Training
    start_train = time.time()
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    end_train = time.time()
    train_time = end_train - start_train

    # Evaluation
    start_test = time.time()
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    end_test = time.time()
    test_time = end_test - start_test

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    return {
        'Split Ratio': f"{int((1-split_ratio)*100)}:{int(split_ratio*100)}",
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'Train Time (s)': train_time,
        'Test Time (s)': test_time
    }

# === Run for different split ratios ===
split_ratios = [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
results = []

for ratio in split_ratios:
    print(f"\n🚀 Running for Train-Test split = {int((1-ratio)*100)}:{int(ratio*100)}")
    res = train_evaluate(ratio)
    results.append(res)

# Save results to Excel
df_results = pd.DataFrame(results)
df_results.to_excel('cnn_split_results.xlsx', index=False, engine='openpyxl')
print("\n✅ Results saved to cnn_split_results.xlsx")


In [ ]:
### ResNet-1D Classifier

In [ ]:
from scipy.signal import savgol_filter
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- SNV normalization function ---
def snv(input_data):
    """
    Standard Normal Variate (SNV) normalization
    """
    return (input_data - np.mean(input_data, axis=1, keepdims=True)) / np.std(input_data, axis=1, keepdims=True)

# Load and preprocess data
df = pd.read_csv('Dataset _Raman.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

# ✅ Apply Savitzky–Golay first derivative (25 points, polyorder=2)
X_sg = savgol_filter(X.values, window_length=25, polyorder=2, deriv=1, axis=1)

# ✅ Apply SNV normalization
X_snv = snv(X_sg)

# ✅ Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snv)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Define Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, downsample=False):
        super(ResidualBlock, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=1, padding=padding)
        self.bn2 = nn.BatchNorm1d(out_channels)

        if downsample or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        out = self.relu(out)
        return out

# Define ResNet1D
class ResNet1D(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ResNet1D, self).__init__()
        self.layer1 = ResidualBlock(1, 32, kernel_size=7, stride=2, downsample=True)
        self.layer2 = ResidualBlock(32, 64, kernel_size=5, stride=2, downsample=True)
        self.layer3 = ResidualBlock(64, 128, kernel_size=3, stride=2, downsample=True)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x).squeeze(-1)
        x = self.fc(x)
        return x

# Training function
def train_model(model, train_loader, criterion, optimizer, epochs=20):
    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

# Evaluation function
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return all_labels, all_preds

# Simple timing wrapper
def measure_time(func, *args, **kwargs):
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    elapsed_time = end_time - start_time
    return result, elapsed_time

# Loop over train-test ratios
ratios = [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
results = []

for test_size in ratios:
    print(f"\n===== Train:Test = {1-test_size:.2f}:{test_size:.2f} =====")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_encoded, test_size=test_size, random_state=42, stratify=y_encoded
    )

    # Convert to tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # Model setup
    model = ResNet1D(X.shape[1], len(np.unique(y_encoded)))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    # Train with timing
    _, train_time = measure_time(train_model, model, train_loader, criterion, optimizer)

    # Test with timing
    (labels_preds, test_time) = measure_time(evaluate_model, model, test_loader)
    all_labels, all_preds = labels_preds

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    results.append({
        'Train:Test': f"{int((1-test_size)*100)}:{int(test_size*100)}",
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'Train Time (s)': train_time,
        'Test Time (s)': test_time
    })

# Save results to Excel
df_results = pd.DataFrame(results)
df_results.to_excel('results.xlsx', index=False, engine='openpyxl')

print("\n✅ All results saved to 'results.xlsx'")


In [ ]:
### Transformer for Raman 1D Classification


### Transformer for Raman 1D Classification
from scipy.signal import savgol_filter   # ✅ Added
import numpy as np

# === Step 1: Load and preprocess the data ===
df = pd.read_csv('Dataset _Raman.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

# --- SNV normalization function ---
def snv(input_data):
    """
    Standard Normal Variate (SNV) normalization
    """
    return (input_data - np.mean(input_data, axis=1, keepdims=True)) / np.std(input_data, axis=1, keepdims=True)

# ✅ Apply Savitzky–Golay first derivative (25 points, polyorder=2)
X_sg = savgol_filter(X.values, window_length=25, polyorder=2, deriv=1, axis=1)

# ✅ Apply SNV normalization
X_snv = snv(X_sg)

# ✅ Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snv)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ✅ Best hyperparameters
best_lr = 0.0002198064708278251
best_embed_dim = 256
best_nhead = 8
best_ff_dim = 128
best_num_layers = 1
best_batch_size = 32


# === Step 2: Transformer Encoder ===
class TinyTransformer1D(nn.Module):
    def __init__(self, input_len, n_classes, embed_dim, nhead, ff_dim, num_layers):
        super(TinyTransformer1D, self).__init__()
        self.embedding = nn.Linear(input_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=ff_dim,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = x.squeeze(1)  # (batch, seq_len)
        x = self.embedding(x).unsqueeze(1)  # (batch, 1, d_model)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

# === Step 3: Train/Evaluate with different split ratios ===
results = []

split_ratios = [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]

for ratio in split_ratios:
    print(f"\n=== Running for Train-Test Split {int((1-ratio)*100)}:{int(ratio*100)} ===")

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=ratio, random_state=42)

    X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test = torch.tensor(y_test, dtype=torch.long)

    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=best_batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=best_batch_size)

    model = TinyTransformer1D(
        input_len=X_train.shape[-1],
        n_classes=len(le.classes_),
        embed_dim=best_embed_dim,
        nhead=best_nhead,
        ff_dim=best_ff_dim,
        num_layers=best_num_layers
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
    criterion = nn.CrossEntropyLoss()

    # Training
    start_train = time.time()
    for epoch in range(30):  # fewer epochs for speed
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
    end_train = time.time()
    train_time = end_train - start_train

    # Testing
    start_test = time.time()
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            outputs = model(xb)
            preds = outputs.argmax(dim=1).cpu().numpy()
            labels = yb.cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)
    end_test = time.time()
    test_time = end_test - start_test

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted')
    rec = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # Memory usage (approximate)
    mem_used = psutil.Process().memory_info().rss / (1024 ** 2)  # MB

    results.append({
        'Split Ratio': f"{int((1-ratio)*100)}:{int(ratio*100)}",
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Train Time (s)': train_time,
        'Test Time (s)': test_time,
        'Memory Usage (MB)': mem_used
    })

# === Save Results ===
df_results = pd.DataFrame(results)
df_results.to_excel('transformer_split_ratio_results.xlsx', index=False, engine='openpyxl')

print("\n✅ Results saved to transformer_split_ratio_results.xlsx")


In [ ]:
### Autoencoder + Classifier


### Autoencoder + Classifier
from scipy.signal import savgol_filter   # ✅ Added
import numpy as np

# ===============================
# Load dataset
# ===============================
df = pd.read_csv('Dataset _Raman.csv')
X = df.drop(columns=['Algae_Type', 'ID'])
y = df['Algae_Type']

# --- SNV normalization function ---
def snv(input_data):
    """
    Standard Normal Variate (SNV) normalization
    """
    return (input_data - np.mean(input_data, axis=1, keepdims=True)) / np.std(input_data, axis=1, keepdims=True)

# ✅ Apply Savitzky–Golay first derivative (25 points, polyorder=2)
X_sg = savgol_filter(X.values, window_length=25, polyorder=2, deriv=1, axis=1)

# ✅ Apply SNV normalization
X_snv = snv(X_sg)

# ✅ Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_snv)

# Label encode
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)


# ===============================
# Hyperparameters (Best found)
# ===============================
encoding_dim = 128
ae_batch_size = 32
clf_batch_size = 32
dropout_rate = 0.3

# ===============================
# Train-test split ratios to evaluate
# ===============================
split_ratios = [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]  # corresponds to 90:10, 80:20, etc.

results = []

for test_ratio in split_ratios:
    print(f"\n===== Training with split {int((1-test_ratio)*100)}:{int(test_ratio*100)} =====")

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_categorical, test_size=test_ratio, random_state=42
    )

    # ===============================
    # Autoencoder
    # ===============================
    input_dim = X_train.shape[1]
    input_layer = Input(shape=(input_dim,))
    x = Dense(256, activation='relu')(input_layer)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    encoded = Dense(encoding_dim, activation='relu')(x)

    # Decoder
    x = Dense(128, activation='relu')(encoded)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    decoded = Dense(input_dim, activation='linear')(x)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    early_stop = EarlyStopping(patience=10, restore_best_weights=True)

    start_train = time.time()
    autoencoder.fit(
        X_train, X_train,
        epochs=30,
        batch_size=ae_batch_size,
        validation_data=(X_test, X_test),
        callbacks=[early_stop],
        verbose=0
    )
    end_train = time.time()
    ae_train_time = end_train - start_train

    encoder = Model(inputs=input_layer, outputs=encoded)
    X_train_encoded = encoder.predict(X_train)
    X_test_encoded = encoder.predict(X_test)

    # ===============================
    # Classifier
    # ===============================
    classifier = Sequential([
        Dense(128, activation='relu', input_shape=(encoding_dim,)),
        Dropout(dropout_rate),
        Dense(64, activation='relu'),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dense(y_categorical.shape[1], activation='softmax')
    ])

    classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    start_train = time.time()
    classifier.fit(
        X_train_encoded, y_train,
        epochs=30,
        batch_size=clf_batch_size,
        validation_data=(X_test_encoded, y_test),
        callbacks=[early_stop],
        verbose=0
    )
    end_train = time.time()
    clf_train_time = end_train - start_train

    # ===============================
    # Evaluation
    # ===============================
    start_test = time.time()
    y_pred_probs = classifier.predict(X_test_encoded)
    end_test = time.time()
    test_time = end_test - start_test

    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    acc = accuracy_score(y_true_classes, y_pred_classes)
    prec = precision_score(y_true_classes, y_pred_classes, average='weighted', zero_division=0)
    rec = recall_score(y_true_classes, y_pred_classes, average='weighted')
    f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')

    # Memory usage
    mem_usage = psutil.Process().memory_info().rss / (1024**2)  # MB

    results.append({
        'Train:Test': f"{int((1-test_ratio)*100)}:{int(test_ratio*100)}",
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AE_Train_Time(s)': ae_train_time,
        'Classifier_Train_Time(s)': clf_train_time,
        'Test_Time(s)': test_time,
        'Memory_Usage_MB': mem_usage
    })

# ===============================
# Save Results
# ===============================
df_results = pd.DataFrame(results)
df_results.to_excel('AE_classifier_ratio_analysis.xlsx', index=False, engine='openpyxl')
print("\n✅ Results saved to AE_classifier_ratio_analysis.xlsx")
